In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/genai_project"

Mounted at /content/drive


In [2]:
import shutil

shutil.rmtree(f"{BASE_DIR}/data/synthetic_clean/unclassified", ignore_errors=True)

print("❌ removed unclassified")

❌ removed unclassified


In [3]:
import os
import shutil
import random

SOURCE_DIR = f"{BASE_DIR}/data/synthetic_clean"
TARGET_DIR = f"{BASE_DIR}/data/split"

# reproducibility
SEED = 42
random.seed(SEED)

if os.path.exists(TARGET_DIR):
    print("Deleting old split...")
    shutil.rmtree(TARGET_DIR)

print("Creating new split folders...")

train_ratio = 0.7
val_ratio = 0.15

split_counts = {}

for cls in os.listdir(SOURCE_DIR):
    class_path = os.path.join(SOURCE_DIR, cls)
    images = os.listdir(class_path)
    random.shuffle(images)

    n = len(images)
    n_train = int(train_ratio * n)
    n_val = int(val_ratio * n)

    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train+n_val],
        "test": images[n_train+n_val:]
    }

    split_counts[cls] = {}

    for split in splits:
        os.makedirs(f"{TARGET_DIR}/{split}/{cls}", exist_ok=True)

        for img in splits[split]:
            src = os.path.join(class_path, img)
            dst = f"{TARGET_DIR}/{split}/{cls}/{img}"
            shutil.copy(src, dst)

        # 📊 שמירת כמות
        split_counts[cls][split] = len(splits[split])

print("\n✅ Dataset split complete!\n")

for cls in split_counts:
    print(f"Class: {cls}")
    for split in ["train", "val", "test"]:
        print(f"  {split}: {split_counts[cls][split]}")
    print()

total = sum(split_counts[cls].values())
print(f"  total: {total}")

Creating new split folders...

✅ Dataset split complete!

Class: clean
  train: 49
  val: 10
  test: 11

Class: empty
  train: 49
  val: 10
  test: 11

Class: finished_leftovers
  train: 49
  val: 10
  test: 11

Class: full
  train: 49
  val: 10
  test: 11

  total: 70


In [4]:
print(os.listdir(f"{BASE_DIR}/data/split/train"))

['clean', 'empty', 'finished_leftovers', 'full']
